In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import statsmodels.api as sm
import seaborn as sns
import bambi as bmb
import arviz as az
import scoringrules as sr

pa_gs = pd.read_csv('data/pa_data.csv')
temps = pd.read_csv("data/pa_temps_supplement.csv")
#combining main csv dataset with supplementary temperature data csv
pa_gs = pa_gs.merge(temps, on=["station_id", "year"], how="left")

#limiting dataset to just time interval of interest
pa_gs = pa_gs[pa_gs['year'] >= 1960]

#splitting dataframe in preparation for lagging of variable data
pa_gs_yr_gsl = pa_gs[['year', 'growing_season_length', 'last_spring_frost_doy', 'first_fall_frost_doy', 'station_id']]
pa_gs_non_gsl = pa_gs.drop(['growing_season_length', 'last_spring_frost_doy', 'first_fall_frost_doy'], axis = 1)

#creating lagged dataset where variable data is paired with GSL
pa_gs_lagged = pa_gs_non_gsl.copy(deep=True)
pa_gs_lagged['year'] += 1
pa_gs_lagged_vars = pa_gs_lagged.merge(pa_gs_yr_gsl, on=["station_id", "year"], how = "left")

In [2]:
def remove_nulls_simple(data_subset, variables):
        # Drop rows where any of the specified variable columns have nulls
        data_subset_cleaned = data_subset.dropna(subset=variables, ignore_index=True)
        # Subsets the cleaned dataframe back into only chosen variables
        vars_cleaned = data_subset_cleaned[variables]
        return vars_cleaned

def list_to_str(list_name):
    string = " + ".join(list_name)
    return string
    
def bayesian_model_unstandardized(data, target_var_name, cov_list):

    cov_str = list_to_str(cov_list)
    cov_list_target_var = list(cov_list) + [target_var_name] + ['year', 'station_id']
    data = remove_nulls_simple(data, cov_list_target_var)
    
    formula = bmb.Formula(f"{target_var_name} ~ {cov_str} + year",
                      f"sigma ~ {cov_str} + year")
    model = bmb.Model(formula, data=data, dropna = True)
    output = model.fit(idata_kwargs={'log_likelihood':True})

    resid = data[target_var_name] - model.predict(output, inplace=False).posterior["mu"].mean(("chain", "draw")).values
    
    return model, output, resid


all_covs_list = ['dtr_annual', 'dtr_spring', 'tmean_spring','tmean_fall','latitude','longitude','tmax_annual','oni_annual',
                'nao_annual','pna_annual','amo_annual','sst_north_atlantic','pwat_station','dewpoint_station',
                'soil_moisture_station','cloud_cover_station','evaporation_station',
                'sst_gulf_mexico','pwat_southeast_us','dewpoint_2m_southeast_us','soil_moisture_southeast_us',
                'cloud_cover_southeast_us','evaporation_southeast_us']

cleaned_pa_gs_lagged = remove_nulls_simple(pa_gs_lagged_vars, all_covs_list + ['growing_season_length','year','station_id'])

In [3]:
lagged_model, lagged_output, lagged_resid = bayesian_model_unstandardized(pa_gs_lagged_vars, 'growing_season_length', all_covs_list)

                                                            Grad                                                  
  Progress               Draw        Divergen…   Step size   evals       Speed                Elapsed    Remaini…  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.130       31          139.28 draws/s       0:00:14    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.125       31          127.99 draws/s       0:00:15    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.117       63          121.96 draws/s       0:00:16    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.165       31          144.45 draws/s       0:00:13    0:00:00

/home/reu/.venv/lib/python3.14/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 22 seconds.


In [6]:
dtr_list = ["dtr_annual"]
lagged_model_dtr_only, lagged_output_dtr_only, lagged_resid_dtr_only = bayesian_model_unstandardized(pa_gs_lagged_vars, 'growing_season_length', dtr_list)

                                                            Grad                                                  
  Progress               Draw        Divergen…   Step size   evals       Speed                Elapsed    Remaini…  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.826       3           708.47 draws/s       0:00:02    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           1.344       3           907.52 draws/s       0:00:02    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.977       3           949.65 draws/s       0:00:02    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.692       7           896.08 draws/s       0:00:02    0:00:00

/home/reu/.venv/lib/python3.14/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 4 seconds.


In [7]:
lagged_lsf_model, lagged_lsf_output, lagged_lsf_resid = bayesian_model_unstandardized(pa_gs_lagged_vars, 'last_spring_frost_doy', all_covs_list)

                                                            Grad                                                  
  Progress               Draw        Divergen…   Step size   evals       Speed                Elapsed    Remaini…  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.189       31          160.50 draws/s       0:00:12    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.185       31          121.85 draws/s       0:00:16    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.177       31          168.77 draws/s       0:00:11    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.208       63          127.24 draws/s       0:00:15    0:00:00

/home/reu/.venv/lib/python3.14/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 22 seconds.


In [8]:
lagged_fff_model, lagged_fff_output, lagged_fff_resid = bayesian_model_unstandardized(pa_gs_lagged_vars, 'first_fall_frost_doy', all_covs_list)

                                                            Grad                                                  
  Progress               Draw        Divergen…   Step size   evals       Speed                Elapsed    Remaini…  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.136       31          124.55 draws/s       0:00:16    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.162       31          119.87 draws/s       0:00:16    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.125       31          113.13 draws/s       0:00:17    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.138       31          120.68 draws/s       0:00:16    0:00:00

/home/reu/.venv/lib/python3.14/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 24 seconds.


In [4]:
def get_prediction_ensembles(
    model,
    output,
    data,
    target_var_name,
    id_cols=("station_id", "year"),
    kind="pps",
    new_data=None,
    ci=0.95,
):
    """
    model : The fitted Bambi model.
    output : The fitted result (model.fit output).
    data : The cleaned data used to fit the model (same row order).
    target_var_name : Name of the outcome column.
    id_cols :Columns that identify each row.
    kind : "pps" = predictions including observation noise or "response_params" = mean (mu) and sigma separately.
    new_data : Optional. Predict for new station/year rows instead of the training rows.
    ci : Width of the credible interval for summary columns.
    """

    # New data or existing data
    if new_data is not None:
        rows = new_data.reset_index(drop=True)
        pred = model.predict(output, kind=kind, data=new_data, inplace=False)
    else:
        rows = data.reset_index(drop=True)
        pred = model.predict(output, kind=kind, inplace=False)

    # Pulling predictive draws out of the InferenceData object.
    if kind == "pps":
        draws = pred.posterior_predictive[target_var_name]
    else:
        draws = pred.posterior["mu"]

    # Flatten the chain and draw dimensions into one "sample" dimension.
    obs_dim = [d for d in draws.dims if d not in ("chain", "draw")][0]
    stacked = draws.stack(sample=("chain", "draw")).transpose(obs_dim, "sample")
    ens = stacked.values  # shape: (n_rows, n_samples)

    # Results
    result = rows[list(id_cols)].copy()
    result["ensemble"] = list(ens)  # each cell holds a numpy array of draws
    result["n_draws"] = ens.shape[1]

    # Attach observed values when predicting on training data
    if new_data is None and target_var_name in rows.columns:
        result["observed"] = rows[target_var_name].values
    
    return result

In [5]:
lagged_gsl_predictions = lagged_model.predict(lagged_output, inplace=False).posterior["mu"].mean(("chain", "draw")).values
lagged_gsl_predictions_series = pd.Series(lagged_gsl_predictions)

all_covs_w_gsl = all_covs_list + ['growing_season_length', 'station_id', 'year']
pa_gs_lagged_select_vars = remove_nulls_simple(pa_gs_lagged_vars, all_covs_w_gsl)
#pa_gs_lagged_no_na_gsl = pa_gs_lagged_no_na['growing_season_length']

pa_gs_lagged_w_preds = pd.concat([pa_gs_lagged_select_vars, lagged_gsl_predictions_series], axis=1)
pa_gs_lagged_w_preds.columns= [*pa_gs_lagged_w_preds.columns[:-1], 'gsl_preds']

pa_gs_gsl_obs = pa_gs_lagged_w_preds.pivot(index='year', columns='station_id', values = 'growing_season_length')

In [6]:
lagged_ensembles = get_prediction_ensembles(lagged_model, lagged_output, cleaned_pa_gs_lagged, "growing_season_length", kind="pps")
#lagged_ensembles.head(5)

/home/reu/.venv/lib/python3.14/site-packages/bambi/models.py:925: FutureWarning: 'pps' has been replaced by 'response' and is not going to work in the future
  warnings.warn(


In [38]:
pa_gs_gsl_ens = lagged_ensembles.pivot(index='year', columns = 'station_id', values = 'ensemble')

pa_gs_gsl_ens_array = pa_gs_gsl_ens.to_numpy()
    
pa_gs_gsl_ens_reshaped = np.empty((64, 176, 4000), dtype=np.float32)
for i in range(64):
    for j in range(176):
        pa_gs_gsl_ens_reshaped[i, j] = pa_gs_gsl_ens_array[i, j]

pa_gs_gsl_ens_reshaped = np.transpose(pa_gs_gsl_ens_reshaped, (0,2,1))

In [37]:
pa_gs_gsl_obs_array = pa_gs_gsl_obs.to_numpy()

In [ ]:
lagged_full_variogram_score = sr.variogram_score(pa_gs_gsl_obs_array,pa_gs_gsl_ens_reshaped)

In [26]:
import inspect

#variogram_source_code = inspect.getsource(sr.variogram_score)
#print(variogram_source_code)
??sr.vs_ensemble

Signature:
sr.vs_ensemble(
    obs: 'Array',
    fct: 'Array',
    w: 'Array' = None,
    m_axis: int = -2,
    v_axis: int = -1,
    *,
    ens_w: 'Array' = None,
    p: float = 0.5,
    estimator: str = 'nrg',
    backend: 'Backend' = None,
) -> 'Array'
Source:   
def vs_ensemble(
    obs: "Array",
    fct: "Array",
    w: "Array" = None,
    m_axis: int = -2,
    v_axis: int = -1,
    *,
    ens_w: "Array" = None,
    p: float = 0.5,
    estimator: str = "nrg",
    backend: "Backend" = None,
) -> "Array":
    r"""Compute the Variogram Score for a finite multivariate ensemble.

    For a :math:`D`-variate ensemble the Variogram Score [1]_ is defined as:

    .. math::
        \text{VS}_{p}(F_{ens}, \mathbf{y})= \sum_{i=1}^{d} \sum_{j=1}^{d} w_{i,j}
        \left( \frac{1}{M} \sum_{m=1}^{M} | x_{m,i} - x_{m,j} |^{p} - | y_{i} - y_{j} |^{p} \right)^{2},

    where :math:`\mathbf{X}` and :math:`\mathbf{X'}` are independently sampled ensembles from from :math:`F`.


    Parameters
    --

In [11]:
obs = np.random.randint(150,200, (64,176))
ens = np.random.randint(150,200, (64,4000,176))

obs_uint8 = obs.astype(np.uint8)
ens_uint8 = ens.astype(np.uint8)

#test_vs = sr.variogram_score(obs_uint8, ens_uint8)

In [16]:
lagged_obs_array_chunk1 = pa_gs_gsl_obs_array[10:13, 10:13]
lagged_ens_array_chunk1 = pa_gs_gsl_ens_reshaped[10:13, 0:4000, 10:13]

In [18]:
vs_score_chunk1 = sr.variogram_score(lagged_obs_array_chunk1,lagged_ens_array_chunk1)

In [19]:
#print(lagged_obs_array_chunk1)
#print(lagged_ens_array_chunk1)
print(vs_score_chunk1)

[nan nan nan]


In [20]:
lagged_obs_array_nonan = pa_gs_gsl_obs_array.dropna

AttributeError: 'numpy.ndarray' object has no attribute 'dropna'

In [13]:
pa_gs_gsl_ens_dropna = pa_gs_gsl_ens.dropna(axis=1)
pa_gs_gsl_obs_dropna = pa_gs_gsl_obs.dropna(axis=1)

In [14]:
lagged_ens_dropna_arr = pa_gs_gsl_ens_dropna.to_numpy()
lagged_obs_dropna_arr = pa_gs_gsl_obs_dropna.to_numpy()

In [18]:
lagged_ens_dropna_arr.shape

(64, 11)

In [20]:
lagged_ens_dropna_reshaped = np.empty((64, 11, 4000), dtype=np.float32)
for i in range(64):
    for j in range(11):
        lagged_ens_dropna_reshaped[i, j] = lagged_ens_dropna_arr[i, j]

lagged_ens_dropna_reshaped = np.transpose(lagged_ens_dropna_reshaped, (0,2,1))

In [28]:
vs_dropna_test = sr.variogram_score(lagged_obs_dropna_arr, lagged_ens_dropna_reshaped)

In [29]:
print(vs_dropna_test)

[ 431.59455092  386.71142116  616.9785559   614.47610042  324.12381521
  474.7214762   196.59713966  772.66497647  337.63397594  688.3459279
  347.98735496  423.79942101  482.37874996  652.36291353  554.93630874
  244.06437723  539.96515374  356.57921864  188.34345848  311.67518823
  360.74035787  554.00928137  275.98961673  434.17672559  467.15171612
  433.00547609  334.66459344  635.201012    564.63470878  601.37721366
  251.31005514  277.61438873  474.75687949  413.21611894  338.26847279
 1177.41446726  322.19490116  322.3216301   397.8520911   305.04717538
  779.62807323  536.60780429  266.60988753  325.18993555  454.45907479
  347.82777613  547.99314978  327.59252079  434.06294732  726.53466438
  328.34904121  532.35797973  379.72528511  374.07802042  432.74003661
  338.56250431  233.72452802  378.80372862  271.23586592  436.64511091
  341.51013249  331.13676267  353.2117068   430.44080956]


In [30]:
mean_vs_test_score = np.mean(vs_dropna_test)

In [31]:
print(mean_vs_test_score)

434.31116112902623


In [35]:
pa_gs_gsl_obs_dropna.describe()

station_id,USC00361705,USC00364214,USC00367477,USC00368073,USC00369298,USW00013739,USW00014737,USW00014777,USW00014778,USW00014860,USW00094823
count,64.000000,64.000000,64.000000,64.000000,64.000000,64.00000,64.000000,64.000000,64.000000,64.000000,64.000000
mean,154.546875,154.546875,133.703125,184.406250,155.500000,215.62500,179.312500,173.156250,171.984375,184.406250,174.859375
std,17.671544,17.990235,18.582873,17.816275,17.023793,19.21846,17.925662,15.568812,16.870649,17.820729,15.611216
min,113.000000,106.000000,102.000000,135.000000,113.000000,170.00000,139.000000,123.000000,123.000000,121.000000,143.000000
25%,144.250000,147.000000,120.750000,174.000000,144.000000,201.50000,166.000000,165.000000,160.750000,174.000000,163.750000
50%,154.000000,158.000000,134.000000,185.500000,159.500000,218.50000,179.000000,171.500000,169.500000,186.500000,172.500000
75%,166.000000,166.250000,148.000000,195.000000,166.000000,227.00000,193.000000,185.250000,184.500000,195.000000,186.250000
max,198.000000,198.000000,176.000000,220.000000,191.000000,255.00000,216.000000,206.000000,218.000000,216.000000,209.000000


#### OK, so what have we learned: 
* sr.variogram_score...
    * provides a variogram score for each cell of the inputted data (which I presume we would take the mean of across all cells to get the variogram score for the whole model)
    * will not take NaN values (spits out a NaN as the variogram score for the cell)
    * absolutely annihalates kernals if you try to put more than ~ 10 x 4000 x 25 dimension arrays into it
* given this, might be difficult to have solid conclusions about the accuracy of our models using these rules... I presume we are going to have to interpolate data where stations are missing it before then running our models again and working from there? or would we be interpolating in the model outputs?